In [1]:
import pandas as pd
from ctgan import CTGAN
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

In [2]:
data = pd.read_csv('weatherTT.csv')
condition_to_level = {
    'Fair': 0.0,
    'Partly Cloudy': 0.5,
    'Mostly Cloudy': 0.8,
    'Cloudy': 1.0,
    'Fair / Windy': 1.2,
    'Partly Cloudy / Windy': 1.2,
    'Mostly Cloudy / Windy': 1.4,
    'Cloudy / Windy': 1.4,
    'Mist': 1.6,
    'Patches of Fog': 1.6,
    'Shallow Fog': 1.8,
    'Haze': 1.8,
    'Smoke': 2.0,
    'Smoke / Windy': 2.0,
    'Light Rain Shower': 2.2,
    'Light Rain Shower / Windy': 2.2,
    'Rain Shower': 2.4,
    'Showers in the Vicinity': 2.4,
    'Light Rain': 2.6,
    'Rain': 2.8,
    'Rain / Windy': 2.8,
    'Thunder': 3.0,
    'Light Rain with Thunder': 3.0,
    'T-Storm': 3.0,
    'T-Storm / Windy': 3.0,
    'Thunder in the Vicinity': 3.0,
    'Rain Shower / Windy': 3.0,
    'Heavy Rain Shower': 3.2,
    'Heavy Rain': 3.4,
    'Heavy Rain / Windy': 3.6,
    'Heavy T-Storm': 3.8,
    'Heavy T-Storm / Windy': 3.8,
    'Squalls': 3.9,
    'Squalls / Windy': 3.9,
    'Widespread Dust': 4.0,
    'Widespread Dust / Windy': 4.0,
    'Sandstorm in the Vicinity': 4.0,
    'Fog': 4.0,
    'Rain and Snow': 4.0
}

data['coverage_level'] = data['coverage'].map(condition_to_level)
data.drop(columns = 'coverage', inplace = True)

In [ ]:
ctgan = CTGAN(epochs=500)
ctgan.fit(data, categorical_columns)
synthetic_data = ctgan.sample(2920)

In [60]:
for column in X.columns:
    stat, p = ks_2samp(data[column], synthetic_data[column])
    print(f"{column}: KS Statistic={stat:.3f}, p-value={p:.3f}")

NameError: name 'X' is not defined

In [ ]:
real_corr = data.corr(numeric_only=True)
synth_corr = synthetic_data.corr(numeric_only=True)

# Correlation difference heatmap
diff_corr = np.abs(real_corr - synth_corr)

plt.figure(figsize=(10, 8))
sns.heatmap(diff_corr, cmap='coolwarm', annot=True)
plt.title("Absolute Correlation Difference (Real vs Synthetic)")
plt.show()

In [5]:
timestamps = pd.date_range(start="2025-05-01 00:00:00", end="2026-04-30 23:00:00", freq="h")
num_hours = len(timestamps)  # should be 8784 including leap year 2026
print(num_hours)

8760


In [6]:
synthetic_weather = ctgan.sample(num_hours)
time_df = pd.DataFrame({
    "datetime": timestamps,
    "Year": timestamps.year,
    "Month": timestamps.month,
    "Day": timestamps.day,
    "time": timestamps.hour + 1,  # Assuming your original time column is 1-24
    "Day of Year": timestamps.dayofyear,
    "Is Daylight": ((timestamps.hour >= 6) & (timestamps.hour <= 18)).astype(int),
})

In [13]:
synthetic_weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2920 entries, 0 to 2919
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Day of Year             2920 non-null   int64  
 1   Year                    2920 non-null   int64  
 2   Month                   2920 non-null   int64  
 3   Day                     2920 non-null   int64  
 4   time                    2920 non-null   int64  
 5   Is Daylight             2920 non-null   int64  
 6   Distance to Solar Noon  2920 non-null   float64
 7   temperature             2920 non-null   int64  
 8   wind_direction          2920 non-null   float64
 9   wind_speed              2920 non-null   int64  
 10  humidity                2920 non-null   int64  
 11  pressure                2920 non-null   int64  
 12  coverage_level          2920 non-null   float64
dtypes: float64(3), int64(10)
memory usage: 296.7 KB


In [26]:
synthetic_weather = synthetic_weather.rename(columns={
    'Is Daylight': 'Is Daylight',
    'wind_direction': 'Wind Direction',
    'wind_speed': 'Average Wind Speed (Period)',
    'humidity': 'Relative Humidity',
    'pressure': 'Average Barometric Pressure (Period)',
    'coverage_level': 'Sky Cover'
})

In [34]:
synthetic_weather = synthetic_weather.rename(columns={
    'temperature': 'Average Temperature (Day)',
    'wind_direction': 'Average Wind Direction (Day)',
})

In [35]:
synthetic_weather.columns

Index(['Is Daylight', 'Distance to Solar Noon', 'Average Temperature (Day)',
       'Wind Direction', 'Average Wind Speed (Period)', 'Relative Humidity',
       'Average Barometric Pressure (Period)', 'Sky Cover'],
      dtype='object')

In [37]:
final_df = pd.concat([time_df, synthetic_weather], axis=1)

In [45]:
final_df = final_df.rename(columns={
    'Wind Direction': 'Average Wind Direction (Day)',
})

In [46]:
final_df.columns

Index(['datetime', 'Year', 'Month', 'Day', 'time', 'Day of Year',
       'Is Daylight', 'Distance to Solar Noon', 'Average Temperature (Day)',
       'Average Wind Direction (Day)', 'Average Wind Speed (Period)',
       'Relative Humidity', 'Average Barometric Pressure (Period)',
       'Sky Cover', 'group'],
      dtype='object')

In [54]:
meta_cols = ['datetime', 'Year', 'Month', 'Day', 'time', 'Day of Year', 'Is Daylight', 'Distance to Solar Noon']
avg_cols = ['Average Temperature (Day)',
            'Average Wind Direction (Day)', 'Relative Humidity',
            'Average Wind Speed (Period)', 'Average Barometric Pressure (Period)', 'Sky Cover']

# Add a grouping column: every 3 rows gets the same group number
final_df['group'] = final_df.index // 3

# Aggregate
aggregated_df = (
    final_df
    .groupby('group')
    .agg({**{col: 'first' for col in meta_cols},
          **{col: 'mean' for col in avg_cols}})
    .reset_index(drop=True)
)

In [55]:
aggregated_df['day_of_year_sin'] = np.sin(2 * np.pi * aggregated_df['Day of Year'] / 365)
aggregated_df['day_of_year_cos'] = np.cos(2 * np.pi * aggregated_df['Day of Year'] / 365)

aggregated_df['hour_sin'] = np.sin(2 * np.pi * aggregated_df['time'] / 24)
aggregated_df['hour_cos'] = np.cos(2 * np.pi * aggregated_df['time'] / 24)

# Step 2: Drop original cyclical features
#aggregated_df.drop(['Day of Year', 'First Hour of Period', 'Year', 'Month', 'Day'], axis=1, inplace=True)

# Step 3: Scale all other features
#cyclical_cols = ['day_of_year_sin', 'day_of_year_cos', 'hour_sin', 'hour_cos']
#numerical_cols =aggregated_df.columns.drop(['day_of_year_sin', 'day_of_year_cos', 'hour_sin', 'hour_cos','Power Generated'])

In [56]:
aggregated_df = aggregated_df.drop(columns = ['Day of Year', 'Month', 'Year', 'Day', 'time'], axis = 1)

In [59]:
aggregated_df.to_csv('12_month_forecast.csv', index = False, encoding = 'utf-8')

In [58]:
aggregated_df.head()

,datetime,Is Daylight,Distance to Solar Noon,Average Temperature (Day),Average Wind Direction (Day),Relative Humidity,Average Wind Speed (Period),Average Barometric Pressure (Period),Sky Cover,day_of_year_sin,day_of_year_cos,hour_sin,hour_cos
0,2025-05-01 00:00:00,1.0,0.750328,78.333333,-0.001642,92.333333,0.000000,29.0,0.802057,0.871706,-0.490029,0.258819,0.965926
1,2025-05-01 03:00:00,1.0,0.249362,89.333333,97.573864,58.666667,13.333333,29.0,1.866382,0.871706,-0.490029,0.866025,0.500000
2,2025-05-01 06:00:00,0.0,0.466626,80.000000,59.991876,84.666667,10.000000,29.0,0.861830,0.871706,-0.490029,0.965926,-0.258819
3,2025-05-01 09:00:00,0.0,0.916130,78.333333,29.970231,89.000000,5.333333,29.0,1.001236,0.871706,-0.490029,0.500000,-0.866025
4,2025-05-01 12:00:00,0.0,0.667953,78.666667,30.001422,86.000000,4.333333,29.0,0.434001,0.871706,-0.490029,-0.258819,-0.965926
